# CoRe-TFM: JMLR Robustness Experiments

This notebook is the execution entry point for the experiments that remain after auditing the frozen bounded benchmark. It deliberately does **not** modify `results/q1_fast_complete_256_v1`. New outputs go to a separate robustness evidence directory.

The paper's working thesis is: **compatibility ≠ calibration ≠ predictive correctness**. The goal of these runs is to establish when reconciliation is statistically justified, not to force a positive result.

## 1. Colab setup and repository checkout
Run this cell first in a GPU Colab runtime. Add a Colab secret named `TABPFN_TOKEN` if fresh TabPFN-3 inference is enabled.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    ROOT = Path('/content/core-tfm')
    DRIVE_BASE = Path('/content/drive/MyDrive/CoRe_TFM_Q1')
else:
    ROOT = Path.cwd()
    DRIVE_BASE = ROOT / 'results'

BRANCH = 'research/reliability-aware-enhancements'
if not (ROOT / '.git').exists():
    subprocess.run(['git','clone','https://github.com/bnssaanirudh/core-tfm.git',str(ROOT)], check=True)
subprocess.run(['git','fetch','origin',BRANCH], cwd=ROOT, check=True)
subprocess.run(['git','checkout',BRANCH], cwd=ROOT, check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[test]','pyyaml','ucimlrepo','tabicl==2.1.1','tabpfn','catboost==1.2.10'], cwd=ROOT, check=True)
sys.path.insert(0, str(ROOT/'src'))
os.chdir(ROOT)
print('repo:', ROOT)

## 2. Audit the frozen evidence before running anything new

In [ ]:
FROZEN = ROOT/'results'/'q1_fast_complete_256_v1'
AUDIT_OUT = ROOT/'results'/'reliability_aware_v1'/'evidence_audit.json'
subprocess.run([sys.executable, str(ROOT/'experiments'/'audit_evidence_package.py'), '--results', str(FROZEN), '--output', str(AUDIT_OUT)], check=True)
audit = json.loads(AUDIT_OUT.read_text())
audit

## 3. Reproduce all archive-derived reliability analyses

In [ ]:
DERIVED = ROOT/'results'/'reliability_aware_v1'
DERIVED.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, str(ROOT/'experiments'/'run_reliability_aware_suite.py'), '--fold-results', str(FROZEN/'fold_results.csv'), '--output', str(DERIVED)], check=True)

## 4. Freeze the new robustness protocol **before** inspecting new comparative scores

The protocol below is intentionally written to disk before the new runs. If you change it after seeing test outcomes, record a dated amendment.

In [ ]:
import yaml
cfg = yaml.safe_load((ROOT/'configs'/'reliability_aware_experiments.yaml').read_text())
ROBUST_RUN_ID = 'core_tfm_jmlr_robustness_v1'
ROBUST = DRIVE_BASE/ROBUST_RUN_ID
ROBUST.mkdir(parents=True, exist_ok=True)
protocol = {
    'run_id': ROBUST_RUN_ID,
    'source_branch': BRANCH,
    'source_commit': subprocess.check_output(['git','rev-parse','HEAD'], cwd=ROOT, text=True).strip(),
    'primary_models': ['tabiclv2','tabpfn3'],
    'datasets': ['anneal','car','credit','customer','diamonds','marketing','mic','nursery','phishing','wine'],
    'seed_robustness': cfg['seed_robustness'],
    'context_size': cfg['context_size'],
    'safe_selective': cfg['safe_selective'],
    'rare_class_sensitivity': cfg['rare_class_sensitivity'],
    'third_tfm': cfg['models']['third_tfm'],
    'outcome_blind_freeze': True,
}
(ROBUST/'ROBUSTNESS_PROTOCOL.json').write_text(json.dumps(protocol, indent=2))
protocol

## 5. Multi-seed robustness matrix

Required design: 5 constrained-sampling seeds × 5 outer folds × 10 datasets × 2 primary TFMs. Keep train/test limits at 256/128 to isolate sampling-seed sensitivity.

**Implementation guardrail:** reuse the data-loading, four-view inference, candidate construction, validation selection and fold checkpoint functions from `CoRe_TFM_Q1_FAST_COMPLETE_256_Colab.ipynb`. The sampling seed must enter the split/sampling functions; do not merely relabel the same split. Each seed must write to its own subdirectory.

In [ ]:
SEEDS = cfg['seed_robustness']['seeds']
seed_plan = []
for seed in SEEDS:
    run_dir = ROBUST/f'seed_{seed}'
    run_dir.mkdir(exist_ok=True)
    seed_plan.append({'seed': seed, 'output': str(run_dir), 'train_limit': 256, 'test_limit': 128, 'folds': 5})
pd = __import__('pandas')
pd.DataFrame(seed_plan)

### Multi-seed acceptance criteria
For each seed, require 10 datasets × 2 TFMs × 5 folds = 100 TFM fold tasks. Report the dataset-blocked Selective-minus-arithmetic effect separately per seed and then with a hierarchical summary. Do not pool 50 dataset-seed values as if independent datasets.

## 6. Context-size sensitivity

Run train sizes 64/128/256/512/1024 with seeds 23/42/71 and five folds where each model supports the requested context. Keep test size and evaluation protocol fixed. If a model imposes a lower practical context cap, record it as an operational limitation rather than silently dropping the cell.

In [ ]:
context_plan = pd.DataFrame([
    {'seed': seed, 'train_size': n, 'folds': 5, 'models': 'tabiclv2;tabpfn3'}
    for seed in cfg['context_size']['seeds']
    for n in cfg['context_size']['train_sizes']
])
context_plan.to_csv(ROBUST/'context_plan.csv', index=False)
context_plan

## 7. Rare-class robustness

The audited preflight shows minimum target-class support ≤5 for **Customer (3), Marketing (2), and Nursery (2)**. Run two complementary analyses: (i) exclude datasets below support thresholds 2/5/10; (ii) evaluate support-adaptive marginal penalties `lambda_c=lambda_max*n_c/(n_c+tau)` using validation-only tuning. Never choose the threshold or `tau` from test NLL.

In [ ]:
from core_tfm.research_extensions import support_adaptive_penalties
supports = pd.DataFrame({'dataset':['customer','marketing','nursery'], 'minimum_support':[3,2,2]})
for tau in cfg['rare_class_sensitivity']['support_adaptive_penalty']['taus']:
    supports[f'lambda_tau_{tau:g}'] = support_adaptive_penalties(supports.minimum_support.to_numpy(), base_lambda=10, tau=tau)
supports

## 8. Safe Selective CoRe

Save **all validation candidate scores**. Compare nested families rather than searching all 48 candidates by default. Complexity penalties must depend only on validation size/family size and pre-frozen `beta`, never on test performance. Primary fallback is arithmetic pooling.

In [ ]:
from core_tfm.research_extensions import complexity_penalty
rows=[]
for nv in cfg['safe_selective']['validation_sizes']:
    for m in [1,4,8,16,48]:
        rows.append({'n_validation':nv,'family_size':m,'penalty':complexity_penalty(nv,m)})
pd.DataFrame(rows)

## 9. Oracle opportunity versus selection regret

For each test fold, compute the non-deployable candidate-family oracle only after all selection choices are frozen. Report:

- opportunity = L(arithmetic) − L(oracle)
- selection regret = L(selected) − L(oracle).

This separates 'there was no useful repair' from 'a useful candidate existed but validation failed to identify it'.

In [ ]:
oracle_file = DERIVED/'oracle_selection_decomposition.csv'
if oracle_file.exists():
    oracle = pd.read_csv(oracle_file)
    display(oracle.groupby('model')[['available_opportunity','selection_regret','selective_minus_arithmetic']].mean())

## 10. Per-view reliability archive and inference dispersion

For each example save direct marginal NLL/Brier, conditional NLL/Brier, per-class support, entropy, factorization TV, marginalization defects and the selected policy. For admissible repeated inference, save the full member predictions and Jensen-Shannon dispersion. Call this **inference instability/predictive dispersion** until correlation with held-out error is demonstrated.

## 11. Third TFM preflight

Do not insert a third TFM into the benchmark merely for model count. First verify: released checkpoint/API, classification probability access, conditional-query feasibility, license compatibility, and practical bounded-context runtime. Record failed candidates too. The config leaves `third_tfm: null` until this preflight succeeds.

## 12. Final evidence gate
A JMLR-facing robustness package should not be marked complete until all required cells have manifests, no unresolved failures, frozen environment metadata, checksums, and dataset-blocked statistical summaries. Fresh robustness results may be added to the manuscripts only after this gate passes.

In [ ]:
required_groups = ['multi_seed','context_size','rare_class','safe_selective','view_reliability']
status = {g: (ROBUST/g/'COMPLETE.json').exists() for g in required_groups}
print(status)
print('JMLR robustness gate:', 'PASS' if all(status.values()) else 'PENDING')